# 04 · 权限处理 & Session Hooks

目标：理解 `on_permission_request` 的决策模型；掌握 6 类 `hooks` 钩子；学会用 `on_user_input_request` 让 agent 主动提问。

## 0. 总览 —— SDK 给你伸手干预 LLM 的 3 类钩子

把 SDK 想象成一条"加工流水线"：用户 prompt 进来 → 经过若干层加工 → LLM 决定调工具 → 又经过若干层加工 → 输出。每一层都可以塞自己的 callback：

```mermaid
flowchart LR
    subgraph User["用户"]
        U["session.send(prompt)"]
    end

    subgraph SDK_Pipeline["SDK 加工流水线 (可干预点用🔶标出)"]
        H1["🔶 on_session_start<br/>(注入初始 context)"]
        H2["🔶 on_user_prompt_submitted<br/>(改 prompt / 加 context)"]
        LLM1["LLM 生成"]
        Decide{"要调工具?"}
        H3["🔶 on_pre_tool_use<br/>(allow / deny / 改 args)"]
        Perm["🔶 on_permission_request<br/>(approve-once / reject)"]
        Tool["执行 tool"]
        H4["🔶 on_post_tool_use<br/>(改 result)"]
        H5["🔶 on_session_end / on_error_occurred"]
        Ask["🔶 on_user_input_request<br/>(LLM 反问真人)"]
    end

    subgraph Out["输出"]
        Reply["assistant.message + session.idle"]
    end

    U --> H1 --> H2 --> LLM1 --> Decide
    Decide -- 否 --> Reply
    Decide -- 是 --> H3 --> Perm --> Tool --> H4 --> LLM1
    LLM1 -. ask_user 工具 .-> Ask -. 答案 .-> LLM1
    Reply --> H5
```

| 干预点 | 触发频率 | 典型用途 |
|---|---|---|
| `on_session_start` | 每个 session **1 次** | 注入 `<available_skills>` / `<workspace_info>` 系统块 |
| `on_user_prompt_submitted` | 每条 user prompt | 抹 tag、加 workspace context、prompt 改写 |
| `on_pre_tool_use` | 每次 tool 调用前 | allow/deny/改 args / 附加 audit context |
| `on_post_tool_use` | 每次 tool 调用后 | 后处理 tool result（脱敏、截断、加注解）|
| `on_permission_request` | 每次需要权限的 tool 调用 | 写文件/shell/url/MCP 的裁决（**v2 必填**）|
| `on_user_input_request` | LLM 主动反问时 | 把 `ask_user` 接到 UI / Slack / `input()` |
| `on_session_end` / `on_error_occurred` | 收尾 / 报错 | 落盘 trace、决定 retry/skip/abort |

> 「Hook 与 Permission 有啥区别」？  
> Hook 是**结构化中间件**——总是被调用、可以**改数据**。  
> Permission 是**安全网关**——只在 tool 需要权限时调用、只能返回 approve/reject。  
> 同一个 tool 调用通常会**先**经过 `on_pre_tool_use`（hook） **再**经过 `on_permission_request`（gate）。


## 0. 公共初始化（Provider、Client、TMP）

下面所有 demo 共用同一个 BYOK provider + 临时目录（避免污染仓库）。先跑这个 cell。

In [ ]:
import os, asyncio, tempfile
from dotenv import load_dotenv
from copilot import CopilotClient
from copilot.session import PermissionHandler, PermissionRequestResult
from copilot.generated.session_events import (
    AssistantMessageData, SessionIdleData,
    ToolExecutionStartData, ToolExecutionCompleteData,
    PermissionRequestKind,
)

load_dotenv()

azure_provider = {
    'type': 'azure',
    'base_url': os.environ['AZURE_OPENAI_ENDPOINT_GPT_5_4'],
    'api_key': os.environ['AZURE_OPENAI_API_KEY_GPT_5_4'],
    'azure': {'api_version': os.getenv('AZURE_OPENAI_API_VERSION_GPT_5_4', '2025-03-01-preview')},
}
MODEL = os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4')
TMP_DIR = tempfile.mkdtemp(prefix='ghcp_perm_hooks_')
print('TMP_DIR =', TMP_DIR)


async def _run(prompt: str, *, on_permission_request, hooks=None,
               on_user_input_request=None, on_event=None, timeout=180.0):
    """统一的 demo runner：建会话 + 注册 handler/hook + 发 prompt + 等 idle。"""
    final = {'text': None}
    done = asyncio.Event()

    def _default_on_event(event):
        match event.data:
            case AssistantMessageData() as d:
                if d.content:
                    final['text'] = d.content
            case SessionIdleData():
                done.set()

    kwargs = dict(
        on_permission_request=on_permission_request,
        model=MODEL,
        provider=azure_provider,
        working_directory=TMP_DIR,
    )
    if hooks is not None:
        kwargs['hooks'] = hooks
    if on_user_input_request is not None:
        kwargs['on_user_input_request'] = on_user_input_request

    async with CopilotClient() as client:
        async with await client.create_session(**kwargs) as session:
            # 自定义 on_event 接 idle 时也要释放 done
            if on_event is None:
                session.on(_default_on_event)
            else:
                def composed(event):
                    on_event(event)
                    if isinstance(event.data, AssistantMessageData) and event.data.content:
                        final['text'] = event.data.content
                    if isinstance(event.data, SessionIdleData):
                        done.set()
                session.on(composed)

            await session.send(prompt)
            await asyncio.wait_for(done.wait(), timeout=timeout)
    return final['text']

## 1. 权限模型

**协议 v2** 起，`on_permission_request` 是 session 必填项（除非你打算用事件外置裁决）。Handler 收到每次工具调用前的请求并返回 `PermissionRequestResult`。

请求 `kind`：

| kind | 含义 |
|---|---|
| `shell` | 执行 shell 命令 |
| `write` | 写 / 编辑文件 |
| `read` | 读取文件 |
| `mcp` | 调用 MCP 工具 |
| `custom-tool` | 调用你注册的工具 |
| `url` | fetch URL |
| `memory` | 访问 session/workspace memory |
| `hook` | 触发已注册的 hook |

结果 `kind`：

| kind | 含义 |
|---|---|
| `approve-once` | 本次放行 |
| `reject` | 拒绝 |
| `user-not-available` | 无用户可裁决（默认拒绝） |
| `no-result` | 仅 v1 协议有效 |

### 1.0 权限决策流程图

`on_permission_request` 的输入永远是 `(request, invocation)`，输出永远是 `PermissionRequestResult(kind=...)`。下面这张图把生产里常见的「**写文件白名单 + shell 全禁 + 其它放行**」策略画出来：

```mermaid
flowchart TD
    Start([CLI 触发 permission.request]) --> Kind{request.kind}
    Kind -- shell --> Reject1["return reject"]
    Kind -- url --> AskURL{URL 在白名单?}
    AskURL -- 否 --> Reject2["return reject"]
    AskURL -- 是 --> Approve1["return approve-once"]
    Kind -- write --> Path{file_name 落在 TMP_DIR 子树?}
    Path -- 否 --> Reject3["return reject"]
    Path -- 是 --> Approve2["return approve-once"]
    Kind -- read --> Approve3["return approve-once"]
    Kind -- "mcp / custom-tool / memory / hook" --> Approve4["return approve-once"]

    style Reject1 fill:#ffdddd
    style Reject2 fill:#ffdddd
    style Reject3 fill:#ffdddd
    style Approve1 fill:#ddffdd
    style Approve2 fill:#ddffdd
    style Approve3 fill:#ddffdd
    style Approve4 fill:#ddffdd
```

> **缓存模式**（Bypass / "always allow"）：自维护一个 `set[(kind, target)]`，命中即静默 `approve-once`，未命中弹 UI 问真人，真人选「session-wide」就加入缓存。这是 SDK 没暴露 `approve-for-session` 时的标准模式（见 [README.md](README.md) §5.3）。


In [ ]:
from copilot.session import PermissionRequestResult
from copilot.generated.session_events import PermissionRequest

def my_permission(request: PermissionRequest, invocation: dict) -> PermissionRequestResult:
    # 例：禁止任意 shell；允许只读和自定义工具；写文件白名单
    kind = request.kind.value
    if kind == 'shell':
        return PermissionRequestResult(kind='reject')
    if kind == 'write':
        path = (request.file_name or '')
        if not path.startswith('/tmp/'):
            return PermissionRequestResult(kind='reject')
    return PermissionRequestResult(kind='approve-once')

### 1.1 实战：观察一次 session 里会出现哪几种 `request.kind`

让模型干 3 件混合的事（读、写、shell），观察 `on_permission_request` 被触发的 kind 分布 + 我们的裁决。

In [ ]:
# 💡 演示：分类放行 / 拒绝 + 收集触发的 kind 分布
permission_log: list[tuple[str, str, str]] = []   # (kind, target, decision)

def smart_permission(request, invocation):
    kind = request.kind.value if request.kind else '?'
    target = (
        request.path
        or request.file_name
        or request.full_command_text
        or request.url
        or '*'
    )
    # 简单策略：写文件只允许 TMP_DIR；shell 命令一律拒绝；其它放行
    if kind == 'shell':
        decision = 'reject'
    elif kind == 'write' and not target.startswith(TMP_DIR):
        decision = 'reject'
    else:
        decision = 'approve-once'
    permission_log.append((kind, target, decision))
    print(f'[🔐] kind={kind:<8} decision={decision:<13} target={target!r}')
    return PermissionRequestResult(kind=decision)

async def run_permission_demo():
    # 让模型做 3 件不同 kind 的事
    reply = await _run(
        f'Do all three INDEPENDENTLY:\n'
        f'1. Use apply_patch to create a file "{TMP_DIR}/ok.txt" with content "hello".\n'
        f'2. Use bash to run "echo this-should-be-rejected".\n'
        f'3. Use view to read "{TMP_DIR}/ok.txt" (after step 1 succeeds).\n'
        f'Briefly report which steps succeeded and which were blocked.',
        on_permission_request=smart_permission,
    )

    print('\n' + '=' * 60)
    print('[📊 权限请求统计]')
    print('=' * 60)
    by_kind = {}
    for k, _, d in permission_log:
        by_kind.setdefault(k, {'approve-once': 0, 'reject': 0}).setdefault(d, 0)
        by_kind[k][d] = by_kind[k].get(d, 0) + 1
    for k, ds in by_kind.items():
        print(f'   kind={k:<8}  {ds}')
    print(f'\n[💬 LLM 最终回复]:\n{reply}')

await run_permission_demo()

Async handler 同样支持（适合需要跨网络请求用户确认的场景）：

```python
async def my_permission(request, invocation):
    decision = await ask_user_over_websocket(request)
    return PermissionRequestResult(kind='approve-once' if decision else 'reject')
```

## 2. Hooks —— session 生命周期

比 middleware 更强：可以**修改**输入/输出，提供额外上下文。

| hook | 用途 |
|---|---|
| `on_session_start` | session 创建/恢复时（`source` ∈ `startup`/`resume`/`new`），可注入初始上下文 |
| `on_user_prompt_submitted` | 用户 prompt 提交后、LLM 调用前，可改 prompt |
| `on_pre_tool_use` | 工具执行前，可 allow/deny/修改参数/补上下文 |
| `on_post_tool_use` | 工具执行后，可改结果 |
| `on_session_end` | 收尾 |
| `on_error_occurred` | 决定 `retry` / `skip` / `abort` |

In [ ]:
async def on_pre_tool_use(input_, invocation):
    print(f'[hook] before tool: {input_["toolName"]} args={input_.get("toolArgs")}')
    return {
        'permissionDecision': 'allow',          # 'allow' | 'deny' | 'ask'
        'modifiedArgs': input_.get('toolArgs'), # 可改
        'additionalContext': 'Internal note: caller is workspace ws-42',
    }

async def on_user_prompt_submitted(input_, invocation):
    prompt = input_['prompt']
    return {'modifiedPrompt': f'[workspace=ws-42]\n{prompt}'}

async def on_session_start(input_, invocation):
    return {'additionalContext': '<available_skills>summarize, translate</available_skills>'}

hooks = {
    'on_session_start': on_session_start,
    'on_user_prompt_submitted': on_user_prompt_submitted,
    'on_pre_tool_use': on_pre_tool_use,
}

### 2.1 实战：在一次 session 里看到全部 6 个 hook 触发

下面这个 cell 注册全部 6 类 hook，每个 hook 打印自己被触发的时刻和它能干啥。你会按时间顺序看到：

```
session_start → user_prompt_submitted → pre_tool_use → post_tool_use → session_end
                                       (出错时还会触发 error_occurred)
```

每个 hook 都返回一个修改字段（在 prompt 前加 `[ws-42]`、给 tool 注入 additionalContext 等），可以亲眼看到 LLM 被"中间件"加工的痕迹。

In [ ]:
# 💡 演示：注册全部 6 类 hook，按时序观察触发，并验证它们对 prompt / tool / 上下文的"加工"效果
import time
from typing import Any

hook_trace: list[tuple[float, str, dict]] = []   # (t_offset, hook_name, summary)
t0 = time.time()

def _log_hook(name: str, summary: dict):
    hook_trace.append((time.time() - t0, name, summary))
    print(f't+{time.time()-t0:6.2f}s  🪝 {name:<25} {summary}')


# ────────── 1. session_start：注入 system-level 上下文 ──────────
async def hook_session_start(input_, invocation):
    _log_hook('session_start', {'source': input_.get('source'), 'cwd': input_.get('cwd', '')[:40] + '…'})
    return {
        # 这一段会注入到 system prompt，等价于"workspace context provider"
        'additionalContext': '<workspace_id>ws-42</workspace_id>\n<available_skills>summarize, translate</available_skills>',
    }


# ────────── 2. user_prompt_submitted：改写用户 prompt ──────────
async def hook_user_prompt_submitted(input_, invocation):
    original = input_['prompt']
    _log_hook('user_prompt_submitted', {'orig_len': len(original)})
    # 在 prompt 最前加上 workspace tag，让 LLM "看到"它
    return {'modifiedPrompt': f'[workspace=ws-42] {original}'}


# ────────── 3. pre_tool_use：每次 tool 调用前 → 注入审计 context ──────────
async def hook_pre_tool_use(input_, invocation):
    tn = input_['toolName']
    _log_hook('pre_tool_use', {'tool': tn, 'args_keys': list((input_.get('toolArgs') or {}).keys())})
    return {
        'permissionDecision': 'allow',
        'additionalContext': f'[audit] tool {tn} approved by hook layer for ws-42',
    }


# ────────── 4. post_tool_use：每次 tool 调用后 → 不修改，仅打 marker ──────────
async def hook_post_tool_use(input_, invocation):
    tn = input_['toolName']
    res = input_.get('toolResult')
    res_len = len(str(res)) if res is not None else 0
    _log_hook('post_tool_use', {'tool': tn, 'result_chars': res_len})
    return None   # 不修改 result


# ────────── 5. session_end：收尾时打 marker ──────────
async def hook_session_end(input_, invocation):
    _log_hook('session_end', {'reason': input_.get('reason')})
    return None


# ────────── 6. error_occurred：出错时决定 retry / skip / abort（demo 里几乎不会触发）──────────
async def hook_error_occurred(input_, invocation):
    _log_hook('error_occurred', {
        'ctx': input_.get('errorContext'),
        'err': str(input_.get('error', ''))[:60],
    })
    return {'errorHandling': 'skip'}


all_hooks = {
    'on_session_start': hook_session_start,
    'on_user_prompt_submitted': hook_user_prompt_submitted,
    'on_pre_tool_use': hook_pre_tool_use,
    'on_post_tool_use': hook_post_tool_use,
    'on_session_end': hook_session_end,
    'on_error_occurred': hook_error_occurred,
}

# 让 LLM 真的去调一个工具，触发 pre/post tool use
reply = await _run(
    'Use the view tool to list this directory: ' + TMP_DIR +
    '. Then in your reply, quote whatever workspace_id / available_skills you see in your context.',
    on_permission_request=PermissionHandler.approve_all,
    hooks=all_hooks,
)

print('\n' + '=' * 60)
print(f'[📊 共 {len(hook_trace)} 个 hook 事件]  顺序: {[n for _,n,_ in hook_trace]}')
print('=' * 60)
print(f'\n[💬 LLM 最终回复]:\n{reply}')

> 注意 cowork_worker 的 `WorkspaceMiddleware` 做的两件事 —— ① 从 user message 解析 `[workspace_id=...]` tag，② 在 LLM 调用前注入 `<available_skills>` 系统块 —— 在 SDK 中分别对应 `on_user_prompt_submitted`（改 prompt / 加 context）与 `on_session_start`（首次注入），见 notebook 06。

## 3. `on_user_input_request` —— agent 主动提问

提供该 handler 后，CLI 会启用 `ask_user` 工具。当 LLM 决定问用户问题时，handler 会被调用，签名是 `(request, invocation) -> {'answer': str, 'wasFreeform': bool}`。

`request` 字段：
- `question: str` —— 问题
- `choices: list[str]`（可选）—— 多选项
- `allowFreeform: bool`（默认 True）—— 是否允许自由文本

### 3.1 实战：让 LLM 真正反问你 + 真人 `input()` 回答

下面这个 cell 注册一个会弹 `input()` 阻塞等真人输入的 handler。我们让 LLM 在生成一份"早餐推荐"前**必须先反问** 2 个问题（饮食偏好 + 是否喝咖啡），然后才给出最终回复。

运行时 VS Code 顶部会弹输入框，你打字 + 回车，LLM 才会继续。

In [ ]:
# 💡 演示：真人交互式 ask_user handler。LLM 反问 → input() → LLM 续答
ask_log: list[tuple[str, str]] = []   # (question, answer)


def interactive_ask_user(request, invocation):
    """每次 LLM 调 ask_user 工具时弹 input() 阻塞等真人回答。"""
    q = request.get('question', '?')
    choices = request.get('choices') or []
    allow_freeform = request.get('allowFreeform', True)

    print('\n' + '─' * 60)
    print(f'🤖 LLM 反问: {q}')
    if choices:
        for i, c in enumerate(choices, 1):
            print(f'   [{i}] {c}')
    hint = ' (可输入选项编号或自由文本)' if (choices and allow_freeform) else (
        ' (请输入选项编号)' if choices else ' (自由文本)'
    )
    print('─' * 60)
    raw = input(f'   👉 你的回答{hint}: ').strip()

    # 如果给的是选项编号，转回选项文本
    answer = raw
    was_freeform = True
    if choices and raw.isdigit() and 1 <= int(raw) <= len(choices):
        answer = choices[int(raw) - 1]
        was_freeform = False

    ask_log.append((q, answer))
    print(f'   → 已回答: {answer!r}  (wasFreeform={was_freeform})\n')
    return {'answer': answer, 'wasFreeform': was_freeform}


# ⭐ 关键提示：明确告诉 LLM 必须先反问 2 个具体问题
reply = await _run(
    'Recommend a breakfast for me. But FIRST use the ask_user tool to ask me '
    'these TWO questions one at a time (do NOT bundle into one):\n'
    '  1) "What is your dietary preference?" with choices ["vegan", "vegetarian", "omnivore"]\n'
    '  2) "Do you want coffee?" with choices ["yes", "no"]\n'
    'After getting BOTH answers, give me a 2-sentence breakfast recommendation '
    'that respects my preferences.',
    on_permission_request=PermissionHandler.approve_all,
    on_user_input_request=interactive_ask_user,
    timeout=600.0,   # 给真人留 10 分钟思考
)

print('\n' + '=' * 60)
print(f'[📊 真人被问了 {len(ask_log)} 次]')
for q, a in ask_log:
    print(f'   Q: {q}\n   A: {a}')
print('=' * 60)
print(f'\n[💬 LLM 最终回复]:\n{reply}')

## 4. UI Elicitation

若 host 支持，可用 `session.ui.confirm / select / input / elicitation(schema=...)` 直接弹原生对话框；先检查 `session.capabilities['ui']['elicitation']`。

## 5. Takeaways

- v2 协议必填 `on_permission_request`；最简选择 `PermissionHandler.approve_all`（参见 §1.1 把 shell 全拒、写文件白名单的实战）
- **Hooks 6 件套**是迁移 `AgentMiddleware` / `ContextProvider` 的官方落点（参见 §2.1 一次会话触发全部 6 个 hook 的实战）
- `on_user_input_request` + `ask_user` 让 LLM 反向问用户；handler 阻塞返回 `{answer, wasFreeform}`（参见 §3.1 真人 `input()` 的实战）
- Hooks 的"加工时机"对比：
  - `on_session_start` → 注入**初始 system 上下文**（只触发一次）
  - `on_user_prompt_submitted` → 改写**每条 user prompt**
  - `on_pre_tool_use` → tool 调用前 allow/deny/改参/补 context
  - `on_post_tool_use` → tool 调用后改 result
  - `on_session_end` / `on_error_occurred` → 收尾 / 错误恢复